In [6]:
import os
os.chdir("..")
import pickle
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

os.makedirs("data/wesad_raw", exist_ok=True)  # put your S2.pkl...S17.pkl files here

In [10]:
os.chdir("C:/journal_project")
print("Now in:", os.getcwd())
print("Contents of data/wesad_raw:", os.listdir("data/wesad_raw") if os.path.exists("data/wesad_raw") else "FOLDER DOES NOT EXIST")

Now in: C:\journal_project
Contents of data/wesad_raw: ['S10.pkl', 'S11.pkl', 'S13.pkl', 'S14.pkl', 'S15.pkl', 'S16.pkl', 'S17.pkl', 'S2.pkl', 'S3.pkl', 'S4.pkl', 'S5.pkl', 'S6.pkl', 'S7.pkl', 'S8.pkl', 'S9.pkl']


In [11]:
import pickle
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, butter, filtfilt

CHEST_FS = 700
WINDOW_SEC = 30
OVERLAP = 0.5

LABEL_MAP = {1: "NORMAL_or_FOCUSED", 2: "STRESSED", 3: "NORMAL_or_FOCUSED"}


def bandpass_ecg(signal, fs, low=5, high=15):
    nyq = fs / 2
    b, a = butter(3, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, signal)


def extract_hrv_from_ecg(ecg_window, fs=CHEST_FS):
    if len(ecg_window) < fs * 10:
        return None

    filtered = bandpass_ecg(ecg_window, fs)
    squared = filtered ** 2

    min_distance = int(fs * 0.3)
    threshold = np.mean(squared) + 1.5 * np.std(squared)
    peaks, _ = find_peaks(squared, height=threshold, distance=min_distance)

    if len(peaks) < 15:
        return None

    rr_ms = np.diff(peaks) / fs * 1000
    hr = len(peaks) / (len(ecg_window) / fs) * 60
    diffs = np.diff(rr_ms)
    rmssd = np.sqrt(np.mean(diffs ** 2)) if len(diffs) > 0 else np.nan
    sdnn = np.std(rr_ms) if len(rr_ms) > 1 else np.nan
    pnn50 = (np.sum(np.abs(diffs) > 50) / len(diffs) * 100) if len(diffs) > 0 else np.nan

    return hr, rmssd, sdnn, pnn50


def process_subject(filepath, subject_id):
    with open(filepath, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    ecg = data["signal"]["chest"]["ECG"].flatten()
    eda = data["signal"]["chest"]["EDA"].flatten()
    temp = data["signal"]["chest"]["Temp"].flatten()
    label = data["label"]

    window_n = WINDOW_SEC * CHEST_FS
    step = int(window_n * (1 - OVERLAP))
    n = min(len(ecg), len(label))

    rows = []
    for start in range(0, n - window_n, step):
        end = start + window_n
        window_labels = label[start:end]
        vals, counts = np.unique(window_labels, return_counts=True)
        dominant = vals[np.argmax(counts)]
        if counts.max() / len(window_labels) < 0.9 or dominant not in LABEL_MAP:
            continue

        hrv_result = extract_hrv_from_ecg(ecg[start:end])
        if hrv_result is None:
            continue
        hr, rmssd, sdnn, pnn50 = hrv_result

        eda_window = eda[start:end]
        temp_window = temp[start:end]

        rows.append({
            "subject": subject_id, "hr": hr, "hrv": rmssd, "sdnn": sdnn, "pnn50": pnn50,
            "eda": np.mean(eda_window), "wrist_temp": np.mean(temp_window),
            "wesad_label": LABEL_MAP[dominant],
        })

    return pd.DataFrame(rows)

In [12]:
files = [f for f in os.listdir("data/wesad_raw") if f.endswith(".pkl")]
all_data = []
for fname in sorted(files):
    subj_id = int(fname.replace("S", "").replace(".pkl", ""))
    print(f"Processing subject {subj_id}...")
    df = process_subject(os.path.join("data/wesad_raw", fname), subj_id)
    all_data.append(df)
    print(f"  {len(df)} windows extracted")

wesad_raw_features = pd.concat(all_data, ignore_index=True)
wesad_raw_features["ground_truth"] = wesad_raw_features["wesad_label"].apply(
    lambda x: "STRESSED" if x == "STRESSED" else "NORMAL"
)
wesad_raw_features.to_csv("outputs/wesad_raw_features.csv", index=False)
print(f"\nTotal: {len(wesad_raw_features)} rows")
print(wesad_raw_features["ground_truth"].value_counts())

Processing subject 10...
  148 windows extracted
Processing subject 11...
  144 windows extracted
Processing subject 13...
  143 windows extracted
Processing subject 14...
  144 windows extracted
Processing subject 15...
  143 windows extracted
Processing subject 16...
  143 windows extracted
Processing subject 17...
  149 windows extracted
Processing subject 2...
  135 windows extracted
Processing subject 3...
  138 windows extracted
Processing subject 4...
  141 windows extracted
Processing subject 5...
  144 windows extracted
Processing subject 6...
  143 windows extracted
Processing subject 7...
  141 windows extracted
Processing subject 8...
  142 windows extracted
Processing subject 9...
  142 windows extracted

Total: 2140 rows
ground_truth
NORMAL      1498
STRESSED     642
Name: count, dtype: int64


In [13]:
for col in ["hr", "hrv", "eda", "wrist_temp"]:
    n_range = (wesad_raw_features[wesad_raw_features.ground_truth == "NORMAL"][col].min(),
               wesad_raw_features[wesad_raw_features.ground_truth == "NORMAL"][col].max())
    s_range = (wesad_raw_features[wesad_raw_features.ground_truth == "STRESSED"][col].min(),
               wesad_raw_features[wesad_raw_features.ground_truth == "STRESSED"][col].max())
    overlap = not (s_range[1] < n_range[0] or s_range[0] > n_range[1])
    print(f"{col}: NORMAL {n_range}  STRESSED {s_range}  Overlap: {overlap}")

hr: NORMAL (np.float64(48.0), np.float64(106.0))  STRESSED (np.float64(64.0), np.float64(152.0))  Overlap: True
hrv: NORMAL (np.float64(4.838322410247003), np.float64(218.90982985111054))  STRESSED (np.float64(1.9437767526087852), np.float64(216.70639540073608))  Overlap: True
eda: NORMAL (np.float64(0.5724526904878162), np.float64(13.002052161807105))  STRESSED (np.float64(1.1796703520275298), np.float64(20.151473708379836))  Overlap: True
wrist_temp: NORMAL (np.float32(28.17887), np.float32(35.093987))  STRESSED (np.float32(31.061766), np.float32(35.63516))  Overlap: True


In [14]:
before = len(wesad_raw_features)

wesad_raw_features = wesad_raw_features[
    (wesad_raw_features["hr"].between(40, 180)) &
    (wesad_raw_features["hrv"].between(2, 250)) &        # widened slightly — real values are landing near the old edges
    (wesad_raw_features["eda"].between(0, 25)) &
    (wesad_raw_features["wrist_temp"].between(25, 40))
].reset_index(drop=True)

after = len(wesad_raw_features)
print(f"Dropped {before - after} implausible rows ({(before-after)/before*100:.1f}%)")
print(f"Remaining: {after} rows")
print(wesad_raw_features["ground_truth"].value_counts())

wesad_raw_features.to_csv("outputs/wesad_raw_features.csv", index=False)

Dropped 1 implausible rows (0.0%)
Remaining: 2139 rows
ground_truth
NORMAL      1498
STRESSED     641
Name: count, dtype: int64


In [15]:
for col in ["hr", "hrv", "eda", "wrist_temp"]:
    n_range = (wesad_raw_features[wesad_raw_features.ground_truth == "NORMAL"][col].min(),
               wesad_raw_features[wesad_raw_features.ground_truth == "NORMAL"][col].max())
    s_range = (wesad_raw_features[wesad_raw_features.ground_truth == "STRESSED"][col].min(),
               wesad_raw_features[wesad_raw_features.ground_truth == "STRESSED"][col].max())
    overlap = not (s_range[1] < n_range[0] or s_range[0] > n_range[1])
    print(f"{col}: NORMAL {n_range}  STRESSED {s_range}  Overlap: {overlap}")

hr: NORMAL (np.float64(48.0), np.float64(106.0))  STRESSED (np.float64(64.0), np.float64(148.0))  Overlap: True
hrv: NORMAL (np.float64(4.838322410247003), np.float64(218.90982985111054))  STRESSED (np.float64(2.041241452319318), np.float64(216.70639540073608))  Overlap: True
eda: NORMAL (np.float64(0.5724526904878162), np.float64(13.002052161807105))  STRESSED (np.float64(1.1796703520275298), np.float64(20.151473708379836))  Overlap: True
wrist_temp: NORMAL (np.float32(28.17887), np.float32(35.093987))  STRESSED (np.float32(31.061766), np.float32(35.63516))  Overlap: True
